# Portfolio Analysis - Data Processing Notebook

## Purpose
This notebook orchestrates the complete data pipeline for portfolio analysis project. It retrieves, cleans, processes, and transforms financial market data to support multi-asset portfolio performance analysis and comparison.

## Data Pipeline Overview
The notebook performs the following key operations:

1. **Data Retrieval**: Downloads historical daily price data from Yahoo Finance for:
   - South African equities (FirstRand, Standard Bank, Naspers, Anglo American, MTN Group)
   - US technology stocks (Apple, Microsoft, Nvidia)
   - Currency exchange rate (USD/ZAR)
   - Time period: 2 years of historical data

2. **Data Cleaning**: 
   - Handles missing values through linear interpolation in excel
   - Filters and validates data quality
   - Standardizes asset naming conventions

3. **Data Transformation**:
   - Converts data between wide and long formats for different analytical needs
   - Creates structured views for portfolio performance comparison
   - Generates growth metrics and daily performance indicators

4. **Output Generation**:
   - Exports processed datasets to CSV format
   - Generates portfolio comparison views
   - Produces performance summary tables

 

In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime
import os

In [2]:
# Define the assets and their corresponding clean names
assets = {
    "FSR.JO": "FirstRand",
    "SBK.JO": "Standard_Bank",
    "NPN.JO": "Naspers",
    "AGL.JO": "Anglo_American",
    "MTN.JO": "MTN_Group",
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "NVDA": "Nvidia",
    "USDZAR=X": "USD_ZAR_Rate"
}

In [3]:
print("Initiating data pull from Yahoo Finance...")

#  Fetching 2 years of daily 'Adjusted Close' prices
raw_data =yf.download(list(assets.keys()), start="2024-01-01")['Close']  

if isinstance(raw_data, pd.Series):
    raw_data = raw_data.to_frame()

# Save Wide Version  
raw_data.to_csv('wide_market_data.csv') 
print(f"Wide Data: {raw_data.shape[1]} columns (Stocks)")

Initiating data pull from Yahoo Finance...


[*********************100%***********************]  9 of 9 completed


Wide Data: 9 columns (Stocks)


missing values were linearly intrapolated in excel. The missing value is is a anverage of the day before and after

In [4]:
# Import clean_wide_market_data.xlsx and convert to long format
print("\nImporting clean_wide_market_data.xlsx...")
PATH = "data/clean_wide_market_data.xlsx"
clean_wide_data = pd.read_excel(PATH, index_col=0)
clean_wide_data.head()



Importing clean_wide_market_data.xlsx...


,AAPL,Apple,AGL.JO,Anglo_American,FSR.JO,FirstRand,MSFT,Microsoft,MTN.JO,MTN_Group,NPN.JO,Naspers,NVDA,Nvidia,SBK.JO,Standard_Bank,USDZAR=X,USD_ZAR_Rate
Date,,,,,,,,,,,,,,,,,,
2024-01-01,183.731293,183.731293,44752.562500,44752.562500,7243.378906,7243.378906,364.589447,364.589447,11521.847656,11521.847656,60997.750000,60997.750000,48.138569,48.138569,20672.369141,20672.369141,18.269600,18.269600
2024-01-02,183.731293,183.731293,44752.562500,44752.562500,7243.378906,7243.378906,364.589447,364.589447,11521.847656,11521.847656,60997.750000,60997.750000,48.138569,48.138569,20672.369141,20672.369141,18.300400,18.300400
2024-01-03,182.355606,182.355606,43227.796875,43227.796875,7010.656250,7010.656250,364.324036,364.324036,11391.939453,11391.939453,61738.894531,61738.894531,47.539940,47.539940,20258.941406,20258.941406,18.555590,18.555590
2024-01-04,180.039673,180.039673,43791.714844,43791.714844,7090.561035,7090.561035,361.709137,361.709137,11300.004883,11300.004883,62556.031250,62556.031250,47.968681,47.968681,20333.837891,20333.837891,18.674299,18.674299
2024-01-05,179.317154,179.317154,43915.511719,43915.511719,7100.549316,7100.549316,361.522308,361.522308,11295.007812,11295.007812,61586.105469,61586.105469,49.067013,49.067013,20503.605469,20503.605469,18.686899,18.686899


In [5]:

filtered_data = clean_wide_data[[ 'FirstRand', 'Standard_Bank', 'Naspers', 'Anglo_American', 'MTN_Group', 'Apple', 'Microsoft', 'Nvidia', 'USD_ZAR_Rate']]
null_counts = filtered_data.isnull().sum()
print("\nNull values in filtered data:")
print(null_counts)
filtered_data.head()



Null values in filtered data:
FirstRand         0
Standard_Bank     0
Naspers           0
Anglo_American    0
MTN_Group         0
Apple             0
Microsoft         0
Nvidia            0
USD_ZAR_Rate      0
dtype: int64


,FirstRand,Standard_Bank,Naspers,Anglo_American,MTN_Group,Apple,Microsoft,Nvidia,USD_ZAR_Rate
Date,,,,,,,,,
2024-01-01,7243.378906,20672.369141,60997.750000,44752.562500,11521.847656,183.731293,364.589447,48.138569,18.269600
2024-01-02,7243.378906,20672.369141,60997.750000,44752.562500,11521.847656,183.731293,364.589447,48.138569,18.300400
2024-01-03,7010.656250,20258.941406,61738.894531,43227.796875,11391.939453,182.355606,364.324036,47.539940,18.555590
2024-01-04,7090.561035,20333.837891,62556.031250,43791.714844,11300.004883,180.039673,361.709137,47.968681,18.674299
2024-01-05,7100.549316,20503.605469,61586.105469,43915.511719,11295.007812,179.317154,361.522308,49.067013,18.686899


In [6]:
#convert to long format
long_data = filtered_data.reset_index().melt(id_vars='Date', var_name='Asset', value_name='Price')
long_data['Date'] = pd.to_datetime(long_data['Date'])
long_data.rename(columns={'Date': 'price_date'}, inplace=True)
long_data.head()

,price_date,Asset,Price
0,2024-01-01,FirstRand,7243.378906
1,2024-01-02,FirstRand,7243.378906
2,2024-01-03,FirstRand,7010.656250
3,2024-01-04,FirstRand,7090.561035
4,2024-01-05,FirstRand,7100.549316


In [7]:

filtered_data.to_csv('data/filtered_wide_market_data.csv', index=False)
long_data.to_csv('data/long_market_data.csv', index=False)